## **0. Environment Setup & Base Data Loading**

In [8]:
import os
import pandas as pd
import numpy as np

In [9]:
# Root directory of the project
ROOT_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT_DIR, 'data')
RAW_DATA_DIR = os.path.join(DATA_DIR, 'raw_data')

In [10]:
# Read the CSV file into a DataFrame
df_ames = pd.read_csv(os.path.join(RAW_DATA_DIR, 'AmesHousing.csv'))
df_ames.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


## **1. Temporal Feature Derivation (Age & Remodel Timelines)**
**Business Insight:** Derived `Age At Sale` and `Years Since Remodel` to capture the physical age of the property. Negative values are intentionally retained as they perfectly represent "off-plan" (pre-construction) transactions where the deal was closed before construction/remodeling was completed.

In [4]:
# Define a column for the age at sale
df_ames['Age At Sale'] = df_ames['Yr Sold'] - df_ames['Year Built']
print(df_ames[['Order', 'Year Built', 'Yr Sold', 'Age At Sale']].head()) 
# There is 1 house that was sold before it was built, order 2181, seems like just bought the land and sold it before the house was built, checking the data for this house

# Define a column for the number of years since the last remodel
df_ames['Years Since Remodel'] = df_ames['Yr Sold'] - df_ames['Year Remod/Add']
print(df_ames[['Order', 'Year Remod/Add', 'Yr Sold', 'Years Since Remodel']].head()) 
# There are 3 houses that were sold before they were remodeled, orders 1703, 2181, and 2182, checking the data for these houses
# Seems like the house was sold before it was remodeled (after all the deal was made, just the required record for the construction of the house itself and do not have any effect on the sale price)


   Order  Year Built  Yr Sold  Age At Sale
0      1        1960     2010           50
1      2        1961     2010           49
2      3        1958     2010           52
3      4        1968     2010           42
4      5        1997     2010           13
   Order  Year Remod/Add  Yr Sold  Years Since Remodel
0      1            1960     2010                   50
1      2            1961     2010                   49
2      3            1958     2010                   52
3      4            1968     2010                   42
4      5            1998     2010                   12


## **2. Macroeconomic Integration (Mortgage Rates)**
**Business Insight:** Injected historical 30-year fixed mortgage rates mapped to the exact month and year of each transaction. Real estate is highly sensitive to the cost of borrowing; capturing this macroeconomic indicator allows the model to account for external financial pressures that directly impact buyer purchasing power and housing liquidity.

In [ ]:
# Macro Indicator 
# Calculate the average mortgage rate for each month in the dataset
df_mortgage = pd.read_csv(os.path.join(RAW_DATA_DIR, 'MortgageRates.csv'))
# Convert the weekly mortgage rates to monthly mortgage rates by taking the average of the weekly rates for each month
df_mortgage['Year'] = pd.to_datetime(df_mortgage['Week']).dt.year
df_mortgage['Month'] = pd.to_datetime(df_mortgage['Week']).dt.month
df_mortgage = df_mortgage.groupby(['Year', 'Month'])['U.S. 30 yr FRM'].mean().reset_index()

# Define a column for the mortgage rate at the time of sale
df_ames['Mortgage Rate'] = df_ames.apply(lambda row: df_mortgage[(df_mortgage['Year'] == row['Yr Sold']) & (df_mortgage['Month'] == row['Mo Sold'])]['U.S. 30 yr FRM'].values[0] if not df_mortgage[(df_mortgage['Year'] == row['Yr Sold']) & (df_mortgage['Month'] == row['Mo Sold'])].empty else np.nan, axis=1)
print(df_ames[['Order', 'Yr Sold', 'Mo Sold', 'Mortgage Rate']].head())

   Order  Yr Sold  Mo Sold  Mortgage Rate
0      1     2010        5         4.8875
1      2     2010        6         4.7375
2      3     2010        6         4.7375
3      4     2010        4         5.0980
4      5     2010        3         4.9675


## **3. Synthetic Liquidity Metric (Days on Market)**

**Business Insight:** Engineered `Days on Market` (DOM) as a proxy for housing liquidity. This mathematically models the real-world "U-shaped" market behavior where mass-market properties (average quality, ~1,500 Sq.Ft) sell the fastest, while highly deficient homes and ultra-luxury niche estates suffer from prolonged market exposure. Capping the area strictly prevents unrealistic exponential explosions.

**Mathematical Function Applied:**
$$DOM = 30 + 2 \times (OverallQual - 5.5)^2 + 10 \times \left( \frac{\min(GrLivArea, 3000) - 1500}{500} \right)^2 + \epsilon$$

*(Where: Base DOM = 30 days, and random noise $\epsilon \in [-10, 10]$)*

In [6]:
# Generate data for Days of Market (DOM) based on Overall Quality and Above Grade Living Area (Gr Liv Area)
# DOM Function: 30 (base DOM) + 2 * (Overall Quality - 5.5 (mass market quality))**2 + 10 * (Gr Liv Area - 1500 (mass market area)/500)**2 + Noise
np.random.seed(42)  # For reproducibility
noise_dom = np.random.randint(-10, 11, size=df_ames.shape[0])  # Random noise between -10 and 10

penalty_qual = 2 * (df_ames['Overall Qual'] - 5.5)**2
# Cap the penalty for Above Grade Living Area to avoid extreme values (max penalty when Gr Liv Area is 3000)
penalty_area = 10 * (np.clip(df_ames['Gr Liv Area'], a_min=None, a_max=3000) - 1500)**2 / 500**2
# Calculate Days on Market (DOM) based on the defined function and round to the nearest integer
df_ames['Days on Market'] = 30 + penalty_qual + penalty_area + noise_dom
df_ames['Days on Market'] = np.round(df_ames['Days on Market']).astype(int)

df_ames['Days on Market'] = df_ames['Days on Market'].apply(lambda x: max(1, x))  # Ensure DOM is at least 1

print(df_ames[['Order', 'Overall Qual', 'Gr Liv Area', 'Days on Market']].head())
# Check for 5 largest DOM values to see if they are reasonable
print(df_ames.nlargest(5, 'Days on Market')[['Order', 'Overall Qual', 'Gr Liv Area', 'Days on Market', 'SalePrice']])

df_ames

   Order  Overall Qual  Gr Liv Area  Days on Market
0      1             6         1656              27
1      2             5          896              54
2      3             6         1329              36
3      4             7         2110              49
4      5             5         1629              28
      Order  Overall Qual  Gr Liv Area  Days on Market  SalePrice
2666   2667            10         3608             168     475000
1498   1499            10         5642             162     160000
2445   2446            10         3627             162     625000
2336   2337            10         2945             160     438780
2330   2331            10         3390             158     545224


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice,Age At Sale,Years Since Remodel,Mortgage Rate,Days on Market
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,5,2010,WD,Normal,215000,50,50,4.8875,27
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,6,2010,WD,Normal,105000,49,49,4.7375,54
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,12500,6,2010,WD,Normal,172000,52,52,4.7375,36
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,4,2010,WD,Normal,244000,42,42,5.0980,49
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,3,2010,WD,Normal,189900,13,12,4.9675,28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,...,0,3,2006,WD,Normal,142500,22,22,6.3240,37
2926,2927,923276100,20,RL,NaN,8885,Pave,NaN,IR1,Low,...,0,6,2006,WD,Normal,131000,23,23,6.6820,46
2927,2928,923400125,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,...,700,7,2006,WD,Normal,132000,14,14,6.7625,43
2928,2929,924100070,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,...,0,4,2006,WD,Normal,170000,32,31,6.5075,34


## **4. Spatial Feature Engineering (Concentric Ring Mapping)**

**Business Insight:** Engineered a `Distance to Center` feature to capture spatial pricing dynamics using a Concentric Zone Model (Urban Core, Mid-City, Suburban). Since property values in college towns like Ames are heavily anchored by proximity to the university campus, this spatial metric is critical. Gaussian noise was injected to simulate realistic intra-neighborhood variance, preventing the artificial clustering of identical distances.

**Mathematical Function Applied:**
$$ Distance = \max(0.1, \text{BaseDist}_{(\text{Zone})} + \epsilon) $$

*(Where: $\epsilon \sim \mathcal{N}(\mu=0, \sigma=0.3)$ representing intra-neighborhood spatial variation)*


In [7]:
# Concentric Rings Mapping
# Zone 1: Urban Core / Campus (0.5-1.5 miles): OldTown, IDOTRR, BrkSide, SWISU 
# Zone 2: Mid-City (1.5-3 miles): NAmes, CollgCr, Crawfor, Edwards, Sawyer, SawyerW, Mitchel, ClearCr
# Zone 3: Suburban / Outer Ring: NoRidge, NridgHt, StoneBr, Timber, Veenker, Somerst, Gilbert, NWAmes

neighborhood_dist_map = {
    # Zone 1: Urban Core / Campus (0.5-1.5 miles)
    'OldTown': 0.8, 'IDOTRR': 0.9, 'BrkSide': 1.2, 'SWISU': 1.0, 'Blueste': 1.3,

    # Zone 2: Mid-City (1.5-3 miles)
    'NAmes': 2.0, 'CollgCr': 2.2, 'Crawfor': 2.5, 'Edwards': 2.8, 
    'Sawyer': 2.4, 'SawyerW': 2.6, 'Mitchel': 2.9, 'ClearCr': 2.7, 
    'BrDale': 2.1, 'NPkVill': 2.3, 'Blmngtn': 2.5, 
    'MeadowV': 2.5, 'Landmrk': 2.3,

    # Zone 3: Suburban / Outer Ring (3+ miles)
    'NoRidge': 4.5, 'NridgHt': 4.8, 'StoneBr': 4.2, 'Timber': 4.0, 
    'Veenker': 4.3, 'Somerst': 3.5, 'Gilbert': 3.8, 'NWAmes': 3.6, 
    'Greens': 4.1, 'GrnHill': 4.6
}

# Map the neighborhood to its corresponding distance from the city center (ISU campus)
df_ames['Base Distance'] = df_ames['Neighborhood'].map(neighborhood_dist_map)
# Check if there are any neighborhoods that were not mapped (NaN values)
unmapped_neighborhoods = df_ames[df_ames['Base Distance'].isna()]['Neighborhood'].unique()
if len(unmapped_neighborhoods) > 0:
    print(f"Unmapped neighborhoods found: {unmapped_neighborhoods}")

noise_dict = np.random.normal(loc=0.0, scale=0.3, size=df_ames.shape[0])  # Random noise with mean 0 and std dev 0.3
df_ames['Distance to Center'] = df_ames['Base Distance'] + noise_dict

# Sanity check: Ensure that the distances are non-negative and minimum distance is 0.1 miles
df_ames['Distance to Center'] = np.clip(df_ames['Distance to Center'], a_min=0.1, a_max=None)
df_ames['Distance to Center'] = df_ames['Distance to Center'].round(2)

df_ames.drop('Base Distance', axis=1, inplace=True)  # Drop the intermediate 'Base Distance' column

print(df_ames[['Neighborhood', 'Distance to Center']].head(10))

  Neighborhood  Distance to Center
0        NAmes                1.88
1        NAmes                1.73
2        NAmes                1.90
3        NAmes                1.89
4      Gilbert                3.78
5      Gilbert                3.99
6      StoneBr                4.37
7      StoneBr                4.56
8      StoneBr                3.96
9      Gilbert                3.92


## **5. Data Export**
**Output File:** `data/housing_initial_data.csv`

In [8]:
df_ames

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice,Age At Sale,Years Since Remodel,Mortgage Rate,Days on Market,Distance to Center
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,5,2010,WD,Normal,215000,50,50,4.8875,27,1.88
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,6,2010,WD,Normal,105000,49,49,4.7375,54,1.73
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,6,2010,WD,Normal,172000,52,52,4.7375,36,1.90
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,4,2010,WD,Normal,244000,42,42,5.0980,49,1.89
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,3,2010,WD,Normal,189900,13,12,4.9675,28,3.78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,...,3,2006,WD,Normal,142500,22,22,6.3240,37,2.59
2926,2927,923276100,20,RL,NaN,8885,Pave,NaN,IR1,Low,...,6,2006,WD,Normal,131000,23,23,6.6820,46,2.40
2927,2928,923400125,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,...,7,2006,WD,Normal,132000,14,14,6.7625,43,3.22
2928,2929,924100070,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,...,4,2006,WD,Normal,170000,32,31,6.5075,34,3.16


In [9]:
# Save the modified DataFrame to a new CSV file
output_file_path = os.path.join(DATA_DIR, 'housing_initial_data.csv')
df_ames.to_csv(output_file_path, index=False)